# Fitting a posteriordb model with BlackJAX

`posteriordb_jax.catalog` holds posteriordb's Stan models translated into modelgami models, with their data loaded. Because every model has the same interface, one sampling function runs on all of them. posteriordb also ships reference posterior draws, which we use to check the fit.

Run `./setup_venv.sh` in this folder once and register the kernel it prints; this notebook uses that kernel ("Python (modelgami)").

In [ ]:
import jax
import jax.numpy as jnp
import numpy as np
import blackjax
import arviz as az

from posteriordb_jax import catalog

jax.config.update("jax_enable_x64", True)

catalog.list_models()

## Load a model

The non-centered eight schools model: `theta_trans` are the standardized school effects, `mu` and `tau` the population mean and scale.

In [ ]:
model = catalog.load("eight_schools-eight_schools_noncentered")
print(model.params_pattern)
print("free dimension:", model.free_dim)
model.data

## A sampler for any model

This function only uses the common interface: `log_density_fn`, `random_params` and `flatten_params`. It returns free draws with shape (chain, draw, dimension).

In [ ]:
def nuts(model, key, n_chains=4, n_warmup=1000, n_draws=1000):
    logdensity = model.log_density_fn("joint", params_as="free")

    def run_chain(key):
        init_key, warmup_key, sample_key = jax.random.split(key, 3)
        x_init = model.flatten_params(model.random_params(init_key), to_free=True)
        warmup = blackjax.window_adaptation(blackjax.nuts, logdensity)
        (state, tuned), _ = warmup.run(warmup_key, x_init, num_steps=n_warmup)
        step = blackjax.nuts(logdensity, **tuned).step

        def one_step(state, key):
            state, info = step(key, state)
            return state, (state.position, info.is_divergent)

        _, (positions, divergent) = jax.lax.scan(one_step, state, jax.random.split(sample_key, n_draws))
        return positions, divergent

    free_draws, divergent = jax.jit(jax.vmap(run_chain))(jax.random.split(key, n_chains))
    return free_draws, int(divergent.sum())

In [ ]:
free_draws, n_divergent = nuts(model, jax.random.PRNGKey(0))
print("free draws:", free_draws.shape, "  divergences:", n_divergent)

idata = model.to_arviz(free_draws, params_as="free")
az.summary(idata)

## Compare with posteriordb's reference draws

posteriordb stores reference draws by Stan name. For this model they are of `mu`, `tau` and the transformed parameter `theta` (the school effects), so we compute `theta` from our draws too. (For models whose reference draws cover all the parameters, `catalog.reference_draws` returns them already folded, as in the next section.)

In [ ]:
from posteriordb_jax.io import load_reference_draws
from posteriordb_jax.stan_names import folded_draws_from_posteriordb
import paragami as pg

draws = jax.vmap(jax.vmap(lambda x: model.fold_params(x, from_free=True)))(free_draws)
draws["theta"] = draws["theta_trans"] * draws["tau"][..., None] + draws["mu"][..., None]

reference_pattern = pg.PatternDict(theta=pg.NumericVectorPattern(8), mu=pg.NumericScalarPattern(),
                                   tau=pg.NumericScalarPattern(lb=0.0))
reference = folded_draws_from_posteriordb(reference_pattern,
                                          load_reference_draws("eight_schools-eight_schools_noncentered"))

for name in ["mu", "tau", "theta"]:
    ref, ours = np.asarray(reference[name]), np.asarray(draws[name])
    print(f"{name:12s} mean  reference {np.round(ref.mean(axis=(0, 1)), 2)}\n{'':12s}       NUTS      {np.round(ours.mean(axis=(0, 1)), 2)}")

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(9, 3))
for ax, name in zip(axes, ["mu", "tau"]):
    ax.hist(np.asarray(reference[name]).ravel(), bins=60, density=True, alpha=0.5, label="posteriordb reference")
    ax.hist(np.asarray(draws[name]).ravel(), bins=60, density=True, histtype="step", lw=2, label="BlackJAX NUTS")
    ax.set_title(name)
axes[0].legend(); plt.tight_layout(); plt.show()

## The same code on several models

Because the interface is shared, a method can be run across the catalog. Here we fit a few models and compare posterior means to the reference draws, measured in reference standard deviations.

In [ ]:
names = ["kidiq-kidscore_momiq", "earnings-logearn_height", "mesquite-logmesquite_logvolume", "kilpisjarvi_mod-kilpisjarvi"]
for name in names:
    m = catalog.load(name)
    free_draws, n_div = nuts(m, jax.random.PRNGKey(1), n_warmup=500, n_draws=500)
    ours = jax.vmap(jax.vmap(lambda x: m.fold_params(x, from_free=True)))(free_draws)
    ref = catalog.reference_draws(name)
    z = max(float(np.max(np.abs(np.asarray(ours[k]).mean(axis=(0, 1)) - np.asarray(ref[k]).mean(axis=(0, 1)))
                         / np.asarray(ref[k]).std(axis=(0, 1)))) for k in ref)
    print(f"{name:35s} free dim {m.free_dim:2d}   divergences {n_div:3d}   max |mean difference| / sd = {z:.3f}")

For exchangeable models the ArviZ export includes the pointwise log likelihood, so LOO is one call.

In [ ]:
m = catalog.load("kidiq-kidscore_momiq")
free_draws, _ = nuts(m, jax.random.PRNGKey(2), n_warmup=500, n_draws=500)
az.loo(m.to_arviz(free_draws, params_as="free"))